# 6주차 Basic RAG 질의응답 실습

**GitHub 데이터 → Document → Chunking → Embedding → Chroma → Retrieval → LLM**

GitHub의 `6w/data` 폴더에 저장된 PDF, DOCX, TXT 문서를 자동으로 가져와 실습합니다.

`_____` 부분을 채운 뒤 위에서부터 순서대로 실행하세요.


## 1. 라이브러리 설치

RAG 실습에 필요한 LangChain, Chroma, 임베딩, PDF/DOCX 로더 관련 라이브러리를 설치합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 1. 라이브러리 설치
!pip -q install -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers pypdf docx2txt transformers accelerate sentencepiece


**확인:** 설치가 오류 없이 완료되면 다음 단계로 진행합니다.


## 2. GitHub에서 실습 데이터 가져오기

GitHub 저장소의 `6w/data` 폴더에 있는 실습 문서를 Colab으로 가져옵니다.

저장소는 한 번만 clone하며, 이미 가져온 경우에는 다시 다운로드하지 않습니다.


In [ ]:
# 2. GitHub 저장소에서 실습 데이터 가져오기

import os

REPO_PATH = "/content/2026-ai-sw-convergence"
DATA_PATH = REPO_PATH + "/6w/data"

if not os.path.exists(REPO_PATH):
    !git clone -q https://github.com/hjlee-crypto/2026-ai-sw-convergence.git

print("실습 데이터 준비 완료")
print("데이터 경로:", DATA_PATH)


**확인:** GitHub 저장소가 정상적으로 복제되고 `6w/data` 경로가 출력되는지 확인합니다.


## 3. GitHub의 PDF, DOCX, TXT 문서 불러오기

`data` 폴더의 파일을 확장자에 따라 알맞은 Loader로 읽습니다.

- PDF → `PyPDFLoader`
- DOCX → `Docx2txtLoader`
- TXT → `TextLoader`

서로 다른 파일 형식도 모두 LangChain의 `Document` 객체로 변환됩니다.


In [ ]:
# 3. PDF, DOCX, TXT 문서 불러오기

from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader
)

documents = []

for filename in os.listdir(DATA_PATH):
    file_path = os.path.join(DATA_PATH, filename)

    if filename.lower().endswith("_____"):
        loader = PyPDFLoader(file_path)

    elif filename.lower().endswith("_____"):
        loader = Docx2txtLoader(file_path)

    elif filename.lower().endswith("_____"):
        loader = TextLoader(file_path, encoding="utf-8")

    else:
        continue

    documents.extend(loader.load())

print("불러온 Document 수:", len(documents))


**확인:** `Document 수`가 0보다 커야 합니다. PDF는 페이지별로 여러 Document가 생성될 수 있습니다.


## 4. Document 확인

`page_content`에는 문서의 텍스트가, `metadata`에는 파일명과 페이지 등의 출처 정보가 들어 있는지 확인합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 4. Document 내용과 Metadata 확인
for i, doc in enumerate(documents, 1):
    print(f"\n--- Document {i} ---")
    print("Metadata:", doc._____)
    print(doc._____[:700])


**확인:** 문서 본문과 `source`, `page` 등의 Metadata가 정상적으로 출력되는지 확인합니다.


## 5. Chunking

긴 문서를 한 번에 검색하지 않고 의미 있는 작은 단위인 Chunk로 나눕니다. 일부 내용을 겹치게 하여 문맥이 끊기는 것을 줄입니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 5. Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=_____, chunk_overlap=_____)
chunks = splitter.split_documents(_____)
print("Document 수:", len(documents))
print("Chunk 수:", len(chunks))


**확인:** 원본 Document보다 더 작은 단위의 Chunk가 생성됩니다.


## 6. Chunk와 Metadata 확인

분할된 Chunk의 내용과 원본 파일, 페이지 정보가 유지되는지 확인합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 6. Chunk와 출처 확인
for i, chunk in enumerate(chunks[:8], 1):
    print(f"\n--- Chunk {i} ---")
    print("출처:", chunk.metadata.get("_____"))
    print("페이지:", chunk.metadata.get("page", "-"))
    print(chunk.page_content[:500])


**확인:** Chunk가 나뉘어도 원본 파일과 페이지 정보가 함께 유지됩니다.


## 7. Embedding

문장과 질문을 의미를 나타내는 숫자 벡터로 변환합니다. 비슷한 의미의 문장은 벡터 공간에서도 가까워집니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 7. Embedding
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="_____"
)
vector = embedding_model.embed_query("인턴십 모집인원은 몇 명인가요?")
print("Embedding 차원:", len(vector))
print(vector[:10])


**확인:** 자연어 질문이 고정 길이의 숫자 벡터로 변환됩니다.


## 8. Chroma Vector Store 생성

각 Chunk의 벡터와 원문, Metadata를 Chroma에 저장하여 의미 기반 검색이 가능하도록 합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 8. Chroma Vector Store
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=_____,
    embedding=_____,
    collection_name="week6_basic_rag"
)
print("Vector Store 생성 완료")


**확인:** 이제 질문과 의미가 가까운 Chunk를 검색할 준비가 완료되었습니다.


## 9. 유사도 검색

질문을 벡터로 변환하고 Vector Store에서 의미가 가장 가까운 Chunk를 검색합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 9. 유사도 검색
question = "품종 탐지에는 어떤 모델을 사용하나요?"

results = vectorstore.similarity_search(
    _____,
    k=_____
)

for i, doc in enumerate(results, 1):
    print(f"\n[검색 결과 {i}]")
    print("출처:", doc.metadata.get("source"))
    print(doc.page_content[:600])


**확인:** 질문과 관련된 공지 내용이 상위 검색 결과에 포함되는지 원문과 비교합니다.


## 10. 여러 질문으로 검색 확인

서로 다른 질문을 입력하여 필요한 문서가 실제로 검색되는지 확인합니다. 아직 LLM은 사용하지 않습니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 10. 여러 질문으로 Retrieval 확인
questions = [
    "제머나이소프트 인턴십에서 AI 관련 어떤 경험을 우대하나요?",
    "IPP 참가 신청서 제출기한은 언제인가요?",
    "IPP 이력서 제출기한은 언제인가요?"
]
for question in questions:
    print("\n질문:", question)
    results = vectorstore.similarity_search(_____, k=_____)
    for doc in results:
        print("출처:", doc.metadata.get("_____"))
        print(doc.page_content[:450])


**확인:** 질문이 달라지면 검색되는 Chunk와 출처도 달라지는지 확인합니다.


## 11. Retriever

Vector Store의 검색 기능을 Retriever 형태로 만들어 이후 RAG 과정에서 쉽게 사용할 수 있도록 합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 11. Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": _____})
docs = retriever.invoke("_____")
for doc in docs:
    print(doc.metadata.get("_____"))
    print(doc.page_content[:500])


**확인:** Retriever도 Vector Store와 동일하게 관련 문서를 반환합니다.


## 12. 오픈소스 LLM 준비

검색된 문서를 읽고 최종 답변을 생성할 오픈소스 언어 모델을 준비합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 12. 오픈소스 LLM
from transformers import pipeline
generator = pipeline(
    "text2text-generation",
    model="_____",
    max_new_tokens=_____
)
print("LLM 준비 완료")


**확인:** `LLM 준비 완료`가 출력되면 답변 생성 모델을 사용할 수 있습니다.


## 13. Basic RAG 구성

질문으로 관련 문서를 검색하고, 검색된 내용을 Context로 만들어 질문과 함께 LLM에 전달합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 13. Basic RAG 함수
def basic_rag(question, k=3):
    docs = vectorstore._____(question, k=k)
    context = "\n\n".join(doc._____ for doc in docs)

    prompt = f"""아래 참고 문서만 근거로 질문에 한국어로 간단하고 정확하게 답하세요.
문서에서 확인할 수 없는 내용은 추측하지 말고
'제공된 문서에서 확인할 수 없습니다.'라고 답하세요.

[참고 문서]
{_____}

[질문]
{_____}

[답변]
"""
    answer = generator(_____, max_new_tokens=160)[0]["generated_text"]
    return answer, docs


**핵심 흐름:** `질문 → Retrieval → Context 구성 → Prompt → LLM → Answer`가 하나의 함수로 연결되었습니다.


## 14. RAG 질의응답 실행

실제 질문을 입력하여 Retrieval과 Generation이 연결된 전체 RAG 흐름을 실행합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 14. RAG 질의응답
question = "제머나이소프트 인턴십 모집인원은 몇 명인가요?"
answer, docs = basic_rag(_____)
print("질문:", question)
print("답변:", answer)
print("\n검색 출처")
for doc in docs:
    print("-", doc.metadata.get("_____"), "/ page:", doc.metadata.get("page", "-"))


**확인:** 답변뿐 아니라 어떤 원본 파일과 페이지가 검색되었는지도 함께 확인합니다.


## 15. 여러 질문으로 RAG 확인

여러 질문을 연속으로 실행하여 질문에 따라 검색 문서와 최종 답변이 달라지는지 확인합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 15. 여러 문서를 대상으로 질문 실행
questions = [
    "Oxford-IIIT Pet 데이터는 몇 개 품종을 다루나요?",
    "YOLO 학습 가중치 파일명은 무엇인가요?",
    "탐지와 분류의 차이는 무엇인가요?",
    "IPP 참가 신청서 제출기한은 언제인가요?"
]

for question in questions:
    answer, docs = basic_rag(_____)

    print("\n질문:", question)
    print("답변:", answer)


**확인:** 답변의 근거가 실제 검색된 문서 내용과 일치하는지 비교합니다.


## 16. 문서에 없는 질문 테스트

업로드한 문서에 없는 내용을 질문하여 RAG가 근거 없는 답변을 생성하는지 확인합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 16. 문서에 없는 질문 테스트
question = "_____"

answer, docs = basic_rag(question)

print("질문:", question)
print("답변:", answer)

for doc in docs:
    print("-", doc.metadata.get("source"))


**생각해보기:** Vector Store는 가장 가까운 문서를 반환할 수 있으므로, 검색 결과가 있다고 해서 반드시 정답의 근거가 있는 것은 아닙니다.


## 17. 자유 질문

마지막으로 원하는 질문을 직접 입력하여 업로드한 문서를 대상으로 질의응답을 수행합니다.

> 코드를 실행하기 전에 이 단계에서 무엇을 처리하는지 먼저 확인하세요.


In [ ]:
# 17. 원하는 질문 직접 입력
question = input("문서에 질문하세요: ")
answer, docs = basic_rag(_____)
print("\n답변:", answer)
print("\n검색 출처")
for doc in docs:
    print("-", doc.metadata.get("_____"), "/ page:", doc.metadata.get("page", "-"))


**실습 완료:** 실제 문서 파일을 외부 지식으로 사용하는 Basic RAG의 전체 과정을 완료했습니다.
